In [ ]:
import numpy as np
import time
import cv2
import os
import zlib
from sdlarch_rl import make
import pygame
from IPython.display import Audio
# from stable_baselines3 import PPO
from sbx import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path
from gymnasium.wrappers import TimeLimit
from stable_baselines3.common.monitor import Monitor
# from sbx.ppo.policies import CnnPolicy

from typing import Callable

import flax.linen as nn
import jax.numpy as jnp
from sbx.ppo.policies import PPOPolicy


class ActorCriticCNN(nn.Module):
    n_actions: int
    n_units: int = 512
    activation_fn: Callable[[jnp.ndarray], jnp.ndarray] = nn.relu

    @nn.compact
    def __call__(self, x: jnp.ndarray):
        x = jnp.transpose(x, (0, 2, 3, 1))
        x = x.astype(jnp.float32) / 255.0
        
        x = nn.Conv(32, kernel_size=(8, 8), strides=(4, 4), padding="VALID")(x)
        x = self.activation_fn(x)
        x = nn.Conv(64, kernel_size=(4, 4), strides=(2, 2), padding="VALID")(x)
        x = self.activation_fn(x)
        x = nn.Conv(64, kernel_size=(3, 3), strides=(1, 1), padding="VALID")(x)
        x = self.activation_fn(x)
        x = x.reshape((x.shape[0], -1))
        
        shared_net = nn.Dense(self.n_units)(x)
        shared_net = self.activation_fn(shared_net)

        action_logits = nn.Dense(self.n_actions)(shared_net)
        
        state_value = nn.Dense(1)(shared_net)
        
        return action_logits, state_value


class CnnPolicy(PPOPolicy):
    def build_network(self) -> None:
        self.network = ActorCriticCNN(
            n_actions=self.action_space.n,
            n_units=self.net_arch[0] if self.net_arch else 512,
            activation_fn=self.activation_fn,
        )

NUM_ENV = 8
SAVE_DIR="./model-nsm-sbx"
TENSORBOARD="./tensorboard-nsm-sbx"
TOTAL_TIMESTEP_NUMB = 500_000_000
CHECK_FREQ_NUMB = 5_000
SAVE_FREQ = CHECK_FREQ_NUMB
MAX_STEPS= 12_000

ENT_COEF = 0.001
n_steps=4096
batch_size=64 * NUM_ENV

SAVE_DIR = Path(SAVE_DIR)
combos = [
    [],
    # run
    ["LEFT", "X"],
    ["RIGHT", "X"],
    
    # jump and run
    ["Y"],
    ["LEFT", "X", "Y"],
    ["RIGHT", "X", "Y"],
    
    #shake and run
    ["R2"],
    ["LEFT", "X", "R2"],
    ["RIGHT", "X", "R2"],
]


def make_env(env_id):
    def _init():
        env = make(
            "NewSuperMarioBros-Wii", 
            env_id=env_id,
            #render_mode="human"
        )
        env.set_buttons(["B", "Y", "SELECT", "START", "LEFT", "RIGHT", "DOWN", "UP","A", "X", "L1", "R1", "L2", "R2", "L3", "R3"])

        env = Monitor(env)

        env = MainDiscretizer(
            env,
            combos,
        )

        env = WarpFrame(env, width=96, height=96)
        # env = WarpFrame(env)
        env = MaxAndSkipEnv(env, skip=4)
        env = TimeLimit(env, MAX_STEPS)

        return env
    return _init

    
# env = make_vec_env(make_env(), n_envs=NUM_ENV)
envs = [make_env(i) for i in range(NUM_ENV)]
env = SubprocVecEnv(envs)
env = VecFrameStack(env, 4, channels_order='last')

latest_model_path = get_latest_model(SAVE_DIR)

if latest_model_path:
    print(f"Loading existent model: {latest_model_path}")
    model = PPO.load(
    # model = RecurrentPPO.load(
        str(latest_model_path), 
        env=env, 
        verbose=0, 
        tensorboard_log=TENSORBOARD, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
    )
    
else:
    print("None finded, starting from zero.")
    model = PPO(CnnPolicy,
    # model = RecurrentPPO('CnnLstmPolicy',
        env, 
        verbose=0, 
        # policy_kwargs=policy_kwargs, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
        tensorboard_log=TENSORBOARD, 
    )

# eval_callback = EvalCallback(
#     eval_env, 
#     best_model_save_path="./logs/best_model",
#     log_path="./logs/results", 
#     eval_freq=5_000,
#     n_eval_episodes=6,
#     deterministic=True
# )

checkpoint_callback=TrainAndLoggingCallback(check_freq=CHECK_FREQ_NUMB, save_path=SAVE_DIR, save_freq=SAVE_FREQ, model=model)
#callback = CallbackList([checkpoint_callback, eval_callback])
callback = CallbackList([checkpoint_callback])

model.learn(total_timesteps=TOTAL_TIMESTEP_NUMB, reset_num_timesteps=False, callback=callback)
model.save("final_nsm")

env.close()

Loading existent model: model-nsm-sbx\best_model_15000
Done Rewards Step Cnt: 99
Done Rewards Step Cnt: 45
Done Rewards Step Cnt: 44
Done Rewards Step Cnt: 70
Done Rewards Step Cnt: 70
Done Rewards Step Cnt: 88
Done Rewards Step Cnt: 71
Done Rewards Step Cnt: 45
Done Rewards Step Cnt: 57
Done Rewards Step Cnt: 42
Done Rewards Step Cnt: 133
Done Rewards Step Cnt: 53
Done Rewards Step Cnt: 40
Done Rewards Step Cnt: 43
Done Rewards Step Cnt: 55
Done Rewards Step Cnt: 49
Done Rewards Step Cnt: 63
Done Rewards Step Cnt: 79
Done Rewards Step Cnt: 78
Done Rewards Step Cnt: 271
Done Rewards Step Cnt: 240
Done Rewards Step Cnt: 81
Done Rewards Step Cnt: 54
Done Rewards Step Cnt: 47
Done Rewards Step Cnt: 45
Done Rewards Step Cnt: 68
Done Rewards Step Cnt: 101
Done Rewards Step Cnt: 211
Done Rewards Step Cnt: 98
Done Rewards Step Cnt: 148
Done Rewards Step Cnt: 193
Done Rewards Step Cnt: 46
Done Rewards Step Cnt: 92
Done Rewards Step Cnt: 88
Done Rewards Step Cnt: 164
Done Rewards Step Cnt: 45
D